In [30]:
import pandas as pd
import numpy as np
import pvlib
import matplotlib.pyplot as plt
import seaborn as sns

We have added a new library called pvlib, which is a solar energy library, it helps as to calculate the position of the sun at any time and place by using our latitude and logitude which we know already from the dataset; 

It helps us to create the features which we can calculate the raw variables


### Load the data and setting up the index

In [31]:
df = pd.read_csv("../../Data/solar2022.csv",skiprows=2)

In [32]:
df['datetime']=pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])

In [33]:
df = df.drop(columns=['Year', 'Month', 'Day', 'Hour', 'Minute'])

In [34]:
df = df.set_index('datetime').sort_index()

We converted the separate time columns (`Year`, `Month`, `Day`, `Hour`, `Minute`) into a single `datetime` column using `pd.to_datetime()`.  

After that, we removed the original time component columns and set `datetime` as the DataFrame index with sorting to keep records in chronological order.

This gives a clean time-series structure for feature engineering and analysis.

In [35]:
df.columns

Index(['Temperature', 'Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI',
       'Cloud Type', 'Relative Humidity', 'Pressure', 'Wind Direction',
       'Wind Speed', 'DHI', 'DNI', 'GHI', 'Solar Zenith Angle',
       'Precipitable Water', 'Surface Albedo', 'Fill Flag', 'Dew Point',
       'Asymmetry', 'Aerosol Optical Depth', 'Alpha'],
      dtype='object')

In [36]:
df.index = df.index.tz_localize('Etc/GMT-1')

We have initalized the index timestamp as nsrdb one and the pvlib we are using the international time format UTC and pvlib is aware of the time zone it automatically convert and do the internal calculation 

### Define the site location

In [37]:
location = pvlib.location.Location(
    latitude=40.97,
    longitude=-4.54,
    altitude=895,
    tz='UTC'
)

we have initalized the longitude, latitude, altitude and setting the time zone to do the calcuation using pvlib

## Time-based features

### Compute solar position

In [38]:
solar_pos = location.get_solarposition(df.index)

This is the core pvlib call for the solar feature engineering. It takes every timestamp in the dataframe and calculates the solar features for that timestamp.It returns a dataframe with the solar features for each timestamp.

In [39]:
# Solar zenith angle (refraction-corrected)
df['solar_zenith'] = solar_pos['apparent_zenith']

In [40]:
# Solar elevation (refraction-corrected)
df['solar_elevation'] = solar_pos['apparent_elevation']

In [41]:
# Extraterrestrial irradiance on horizontal surface
dni_extra = pvlib.irradiance.get_extra_radiation(df.index)
cos_zenith = pvlib.tools.cosd(df['solar_zenith'])
df['ET_irradiance'] = (dni_extra * cos_zenith).clip(lower=0)

In [42]:
mae = (df['Solar Zenith Angle'] - df['solar_zenith']).abs().mean()
print(f'Zenith MAE vs NSRDB: {mae:.3f}°')  # expect < 0.5°

Zenith MAE vs NSRDB: 0.003°


We have used MAE as the evaluation metric for the solar feature engineering from the pvblib and the nsdrb. We have got a MAE is 0.03. So, we can say that the values we calculated are good.

###  Hour of day (cyclical encoding)

In [43]:
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)


We need both sin and cos together because one alone is ambiguous. For example sin gives the same value for hour 2 and hour 10, but when combined with cos they become unique. Together they place every hour as a unique point on a circle so the model understands that hour 23 and hour 0 are only 1 hour apart — not 23 hours apart as raw numbers would suggest.

### Day of year (seasonality encoding)

In [44]:
df['doy_sin'] = np.sin(2 * np.pi * df.index.day_of_year / 365)
df['doy_cos'] = np.cos(2 * np.pi * df.index.day_of_year / 365)


Exactly the same logic as hour encoding but applied to seasonality instead of time of day. This encodes the fact that Dec 31st (day 365) and Jan 1st (day 1) are only 1 day apart in reality even though 365 and 1 are far apart as raw numbers. The model can now correctly understand seasonal transitions across year boundaries.

### Calendar features

In [45]:
df['month']        = df.index.month
df['week_of_year'] = df.index.isocalendar().week.astype(int)
df['day_of_week']  = df.index.dayofweek


Extracts the month number (1–12) from every timestamp. Captures broad seasonal patterns — the model can learn that months 6, 7, 8 (summer) consistently have higher GHI than months 12, 1, 2 (winter).


Captures gradual seasonal transitions at a finer resolution than month. While month jumps in steps of ~30 days, week captures changes every 7 days — more granular for the model to learn from.

Extracts the day of the week as an integer (Monday=0, Tuesday=1 ... Sunday=6). Less important for solar irradiance prediction since the sun follows the same path regardless of the day name, but useful if energy consumption patterns on weekdays vs weekends are relevant to the analysis.

### Validate

In [46]:
print(df[['hour_sin','hour_cos','doy_sin','doy_cos','month','week_of_year','day_of_week']].head(10))

                           hour_sin      hour_cos   doy_sin   doy_cos  month  \
datetime                                                                       
2022-01-01 00:30:00+01:00  0.000000  1.000000e+00  0.017213  0.999852      1   
2022-01-01 01:30:00+01:00  0.258819  9.659258e-01  0.017213  0.999852      1   
2022-01-01 02:30:00+01:00  0.500000  8.660254e-01  0.017213  0.999852      1   
2022-01-01 03:30:00+01:00  0.707107  7.071068e-01  0.017213  0.999852      1   
2022-01-01 04:30:00+01:00  0.866025  5.000000e-01  0.017213  0.999852      1   
2022-01-01 05:30:00+01:00  0.965926  2.588190e-01  0.017213  0.999852      1   
2022-01-01 06:30:00+01:00  1.000000  6.123234e-17  0.017213  0.999852      1   
2022-01-01 07:30:00+01:00  0.965926 -2.588190e-01  0.017213  0.999852      1   
2022-01-01 08:30:00+01:00  0.866025 -5.000000e-01  0.017213  0.999852      1   
2022-01-01 09:30:00+01:00  0.707107 -7.071068e-01  0.017213  0.999852      1   

                           week_of_year

## Lagged Weather Features

#### A lag feature is simply the **past value of a variable**. Instead of asking "what is GHI right now?", we ask "what was GHI 1 hour ago? 3 hours ago? 24 hours ago?" The past strongly predicts the future in time series — if it was sunny 1 hour ago, it is likely still sunny now.

### GHI lags

In [47]:
df['GHI_lag_1h']  = df['GHI'].shift(1)
df['GHI_lag_3h']  = df['GHI'].shift(3)
df['GHI_lag_24h'] = df['GHI'].shift(24)

### GHI Lags
- `GHI_lag_1h` — GHI value from 1 hour ago. Captures the **immediate condition** 
  of the sky — is it currently sunny or cloudy right now?
- `GHI_lag_3h` — GHI value from 3 hours ago. Captures the **short term trend** 
  — is the weather changing or staying stable?
- `GHI_lag_24h` — GHI value from the same hour yesterday. Captures the 
  **repeating daily solar pattern** — what did this hour look like yesterday?



### Temperaturen lags

In [48]:
df['temp_lag_1h']  = df['Temperature'].shift(1)
df['temp_lag_3h']  = df['Temperature'].shift(3)
df['temp_lag_24h'] = df['Temperature'].shift(24)

- `temp_lag_1h` — temperature 1 hour ago
- `temp_lag_3h` — temperature 3 hours ago
- `temp_lag_24h` — temperature at the same hour yesterday

Why we lag temperature:
- Rising temperature during the day often signals **clear skies** and increasing 
  solar radiation
- A sudden temperature drop can indicate **incoming cloud cover**
- Yesterday's temperature at the same hour gives the model a reference for 
  what is **seasonally normal**
- The trend of temperature change carries more predictive information than 
  the current value alone


### Humidity lags

In [49]:
df['humidity_lag_1h']  = df['Relative Humidity'].shift(1)
df['humidity_lag_3h']  = df['Relative Humidity'].shift(3)
df['humidity_lag_24h'] = df['Relative Humidity'].shift(24)


- `humidity_lag_1h` — relative humidity 1 hour ago
- `humidity_lag_3h` — relative humidity 3 hours ago
- `humidity_lag_24h` — relative humidity at the same hour yesterday

Why we lag humidity:
- High humidity often **precedes cloud formation** — if humidity was 90% 
  3 hours ago and is now 95%, clouds may be forming and GHI is likely to drop
- Low humidity means **dry clear air** which allows more solar radiation 
  to reach the surface
- The recent history of humidity change is a stronger signal than 
  the current snapshot alone

### Validate

In [50]:
print(df[['GHI', 'GHI_lag_1h', 'GHI_lag_3h', 'GHI_lag_24h']].head(30))

                           GHI  GHI_lag_1h  GHI_lag_3h  GHI_lag_24h
datetime                                                           
2022-01-01 00:30:00+01:00    0         NaN         NaN          NaN
2022-01-01 01:30:00+01:00    0         0.0         NaN          NaN
2022-01-01 02:30:00+01:00    0         0.0         NaN          NaN
2022-01-01 03:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 04:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 05:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 06:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 07:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 08:30:00+01:00    0         0.0         0.0          NaN
2022-01-01 09:30:00+01:00   18         0.0         0.0          NaN
2022-01-01 10:30:00+01:00  141        18.0         0.0          NaN
2022-01-01 11:30:00+01:00  219       141.0         0.0          NaN
2022-01-01 12:30:00+01:00  296       219.0      

## Rolling statistics


A rolling statistic looks at a **sliding window of recent values** and summarises 
them. Instead of asking "what was GHI 1 hour ago?" like lag features do, rolling 
statistics ask "what has GHI been doing over the last 3, 6, or 24 hours on average?" 
The window slides forward one step at a time across the entire dataset.

---

### Why `.shift(1)` before `.rolling()`?
The `.shift(1)` is critical. Without it the rolling window would include the 
**current row** in its own calculation, which is data leakage — you would be 
using the current GHI value to predict itself.




### Rolling mean 

In [51]:
df['GHI_rollmean_3h']  = df['GHI'].shift(1).rolling(window=3).mean()
df['GHI_rollmean_6h']  = df['GHI'].shift(1).rolling(window=6).mean()
df['GHI_rollmean_24h'] = df['GHI'].shift(1).rolling(window=24).mean()

- `GHI_rollmean_3h` — average GHI over the last 3 hours. Captures the 
  **very recent trend** — is GHI rising or falling right now?
- `GHI_rollmean_6h` — average GHI over the last 6 hours. Captures the 
  **half day pattern** — morning buildup or afternoon decline?
- `GHI_rollmean_24h` — average GHI over the last 24 hours. Captures the 
  **full day average** — what was the overall solar output yesterday?

| Window | Time horizon | What it captures |
|---|---|---|
| `3h` | Last 3 hours | Very recent GHI trend |
| `6h` | Last 6 hours | Half day solar pattern |
| `24h` | Last 24 hours | Full day average |


###  Rolling standard deviation

In [52]:
df['GHI_rollstd_3h']  = df['GHI'].shift(1).rolling(window=3).std()
df['GHI_rollstd_6h']  = df['GHI'].shift(1).rolling(window=6).std()
df['GHI_rollstd_24h'] = df['GHI'].shift(1).rolling(window=24).std()

- `GHI_rollstd_3h` — standard deviation of GHI over the last 3 hours
- `GHI_rollstd_6h` — standard deviation of GHI over the last 6 hours
- `GHI_rollstd_24h` — standard deviation of GHI over the last 24 hours

The rolling standard deviation is a **cloudiness proxy**. Think about it:
- **Clear sunny day** → GHI rises and falls smoothly → **low std**
- **Cloudy day** → GHI jumps up and down as clouds pass → **high std**

This means the model can detect unstable, variable weather conditions 
purely from the recent spread of GHI values — without needing any direct 
cloud measurement.

### Validate

In [53]:
print(df[['GHI','GHI_rollmean_3h','GHI_rollmean_6h','GHI_rollmean_24h',
          'GHI_rollstd_3h','GHI_rollstd_6h','GHI_rollstd_24h']].iloc[20:35])

                           GHI  GHI_rollmean_3h  GHI_rollmean_6h  \
datetime                                                           
2022-01-01 20:30:00+01:00    0         7.666667       138.833333   
2022-01-01 21:30:00+01:00    0         0.000000        93.500000   
2022-01-01 22:30:00+01:00    0         0.000000        37.666667   
2022-01-01 23:30:00+01:00    0         0.000000         3.833333   
2022-01-02 00:30:00+01:00    0         0.000000         0.000000   
2022-01-02 01:30:00+01:00    0         0.000000         0.000000   
2022-01-02 02:30:00+01:00    0         0.000000         0.000000   
2022-01-02 03:30:00+01:00    0         0.000000         0.000000   
2022-01-02 04:30:00+01:00    0         0.000000         0.000000   
2022-01-02 05:30:00+01:00    0         0.000000         0.000000   
2022-01-02 06:30:00+01:00    0         0.000000         0.000000   
2022-01-02 07:30:00+01:00    0         0.000000         0.000000   
2022-01-02 08:30:00+01:00    0         0.000000 

## Feature interactions


A feature interaction combines **two existing features into one** by multiplying 
them together. The idea is that sometimes two variables together tell you something 
that neither one tells you alone. For example cloud cover alone does not tell you 
how much GHI is lost — a cloudy sky at midnight loses nothing. But 
**cloud cover × solar elevation** together tells you exactly how much cloud is 
actually blocking available sunlight.




## Cloud cover Proxy

In [54]:
df['cloud_cover_proxy'] = (df['Clearsky GHI'] - df['GHI']).clip(lower=0) / (df['Clearsky GHI'] + 1)

- `Clearsky GHI` — the theoretical maximum GHI if there were zero clouds
- `Clearsky GHI - GHI` — the difference between what is possible and what 
  actually arrived at the surface
- `.clip(lower=0)` — removes negative values since GHI can occasionally exceed 
  Clearsky GHI due to reflections from clouds
- `/ (Clearsky GHI + 1)` — normalises to a 0–1 ratio. The `+1` avoids 
  division by zero at night when Clearsky GHI = 0

The result is a value between 0 and 1:
- `0` → perfectly clear sky, no clouds blocking the sun
- `1` → completely overcast, all sunlight blocked

This is not an interaction feature yet — it is a new intermediate feature 
we create first to use in the interaction below.


### Cloud cover × solar elevation

In [55]:
df['cloud_x_elevation'] = df['cloud_cover_proxy'] * df['solar_elevation'].clip(lower=0)

- Multiplies `cloud_cover_proxy` by `solar_elevation`
- `.clip(lower=0)` removes negative elevation values at nighttime

This captures the **actual impact of clouds on GHI**:

| Condition | Result |
|---|---|
| High cloud cover + low elevation | Small loss — not much sun anyway |
| High cloud cover + high elevation | Big loss — blocking strong midday sun |
| Zero cloud cover + any elevation | Zero — clear sky, no blocking at all |

Neither cloud cover alone nor elevation alone tells you this — only their 
combination does. This is the core value of a feature interaction.

### Humidity × temperature

In [56]:
df['humidity_x_temp'] = df['Relative Humidity'] * df['Temperature']

- Multiplies `Relative Humidity` by `Temperature`

This captures the **combined atmospheric moisture effect**:
- **High humidity + high temperature** → high absolute moisture content in 
  the air → more scattering of solar radiation → lower GHI
- **High humidity + low temperature** → less moisture impact since cold air 
  holds less water vapour
- **Low humidity + any temperature** → dry clear air → higher GHI

Again neither humidity alone nor temperature alone captures this combined 
atmospheric effect — only their product does.

In [57]:
print(df[['GHI', 'Clearsky GHI', 'cloud_cover_proxy', 
          'cloud_x_elevation', 'humidity_x_temp']].iloc[9:18])

                           GHI  Clearsky GHI  cloud_cover_proxy  \
datetime                                                          
2022-01-01 09:30:00+01:00   18            96           0.804124   
2022-01-01 10:30:00+01:00  141           245           0.422764   
2022-01-01 11:30:00+01:00  219           369           0.405405   
2022-01-01 12:30:00+01:00  296           446           0.335570   
2022-01-01 13:30:00+01:00  464           464           0.000000   
2022-01-01 14:30:00+01:00  272           428           0.363636   
2022-01-01 15:30:00+01:00  335           335           0.000000   
2022-01-01 16:30:00+01:00  203           203           0.000000   
2022-01-01 17:30:00+01:00   23            55           0.571429   

                           cloud_x_elevation  humidity_x_temp  
datetime                                                       
2022-01-01 09:30:00+01:00           5.318710          368.753  
2022-01-01 10:30:00+01:00           6.219015          569.633  
2022-0

In [58]:
df.to_csv('../../Outputs/df_engineered.csv')
print(f'Final dataset shape: {df.shape}')
print(f'Total features: {df.shape[1]}')

Final dataset shape: (8760, 48)
Total features: 48


In [59]:
df_clean = df.dropna()


- `GHI_rollmean_3h` → NaN for the first 3 rows
- `GHI_rollmean_6h` → NaN for the first 6 rows
- `GHI_rollmean_24h` → NaN for the first 24 rows
- `GHI_rollstd_24h` → NaN for the first 24 rows

All NaN rows are dropped at the end of feature engineering with `dropna()`.

In [60]:
print(f'Rows before dropna: {len(df)}')
print(f'Rows after dropna:  {len(df_clean)}')
print(f'Rows dropped:       {len(df) - len(df_clean)}')

Rows before dropna: 8760
Rows after dropna:  8736
Rows dropped:       24


In [67]:
columns=['Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI',
       'Cloud Type', 'Wind Direction',
       'Wind Speed', 'DHI', 'DNI',
       'Precipitable Water', 'Surface Albedo', 'Fill Flag', 'Dew Point',
       'Asymmetry', 'Aerosol Optical Depth', 'Alpha']
df_clean=df_clean.drop(columns, axis=1)

In [68]:
df_clean.shape

(8736, 33)

In [69]:
# Save the clean version
df_clean.to_csv('../../Outputs/df_engineered.csv')
print('Saved successfully.')

Saved successfully.


After feature engineering we have total of 48 features and we have removed the the NAN values becuase we don't need them for our model training and testing. And we save them as new file in the output after the feature engineering.
